# Object Tracking Project — Football Player Tracking
## Part 2: Tracking with the BoT-SORT Algorithm

In the previous notebook (`02-ssd_300_tracking.ipynb`) we trained our detector (SSD) and saved it to `best_model.pth`. Now we move to the second stage of the pipeline: **tracking**.

### Entering the Tracking Stage and MOT

For every frame, the detector produces:

```text
Detection
   │
   ├── Bounding Box
   └── Confidence
   └── Class
```

A tracking algorithm takes this information across frames and links objects together so that each player keeps a persistent ID. Since multiple players move simultaneously in this project, our problem is a **MOT — Multiple Object Tracking** problem.

### Using the BoxMOT Library

Instead of implementing tracking algorithms from scratch, we use the ready-made **BoxMOT** library. This library provides several algorithms, such as ByteTrack, BoT-SORT, DeepSORT, StrongSORT, and BoostTrack.

### Why BoT-SORT?

Across tracking algorithms there's a trade-off between **accuracy**, **computational cost**, and **FPS**. Algorithms that use appearance information (**Appearance / Re-ID Features**) in addition to motion tend to have higher association accuracy but also cost more computationally. In this category, **BoT-SORT** and StrongSORT are both attractive options, but since BoT-SORT's computational cost is roughly half of StrongSORT's while offering comparable accuracy, we use **BoT-SORT** in this notebook.

In the next notebook (`04-ByteTrack.ipynb`) we repeat the same pipeline with **ByteTrack** (which relies only on motion, with no Re-ID) and compare it against this one.


### Suppressing Warnings

We suppress unnecessary warnings to keep the output clean.


In [1]:
import warnings
warnings.filterwarnings("ignore")

### Imports

We import `BotSort` from the `boxmot` library, along with OpenCV (for video handling) and PyTorch/NumPy for working with the detector model.


In [2]:
import cv2
import torch
import numpy as np
from boxmot import BotSort

### Loading the Detector (Trained SSD Model)

We set the path to the SSD model saved in the previous notebook (`best_model.pth`) and load it onto the appropriate `device` (`cuda` or `cpu`).


In [3]:
SSD_MODEL_PATH = "best_model.pth"

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.load(SSD_MODEL_PATH,weights_only=False)

In [5]:
device

device(type='cuda')

### Setting the Model to Inference Mode

We move the model to `device` and set it to `eval()` mode, since at this stage we're only doing inference — no more training is needed.


In [6]:
model.to(device)
model.eval()

SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

## Defining the Tracker

### Re-ID and Backbone

BoT-SORT uses a **Re-ID (feature extractor)** network that takes a cropped image of each object and turns it into a **feature vector**:

```text
Object Crop → Re-ID Network → Feature Vector
```

This feature vector helps the tracker judge how similar two objects look across different frames. This is exactly what enables **ID switch reduction** and **re-identification**: e.g. if a player is briefly occluded by another player and reappears a few frames later, the tracker can compare appearance features and reassign the same ID (instead of creating a new one).

This feature extractor needs a backbone. The BoT-SORT paper uses a ResNet, but this isn't mandatory — other backbones can be chosen too (a lighter model gives more speed but lower feature quality, a heavier model the opposite). The names of usable backbones can be found in the BoxMOT documentation/tutorial (the Re-ID Models section).

### Precision (FP32 vs. FP16)

The `half` parameter controls whether Re-ID computation runs at 32-bit precision (FP32) or 16-bit precision (FP16). FP16 increases speed but slightly reduces computational accuracy; in this notebook `half=False` is set, i.e. we use full precision (FP32).


We use `pathlib` to build the path to the Re-ID weights file.


In [8]:
from pathlib import Path

### Specifying the Device for the Tracker

BoxMOT expects the device as a string (`"0"` for the first GPU, or `"cpu"`).


In [9]:
device_str = "0" if torch.cuda.is_available() else "cpu"

### Creating the Tracker Instance

We create a `BotSort` instance with:

* `reid_weights`: the Re-ID model weights (`osnet_x1_0_msmt17.pt` here)
* `device`: GPU or CPU
* `half=False`: use full precision (FP32)
* `with_reid=True`: enable use of appearance features alongside motion


In [10]:
tracker = BotSort(
    reid_weights=Path('osnet_x1_0_msmt17.pt'),
    device=device_str,
    half=False,
    with_reid=True,
)

2025-08-26 11:51:19.920 | INFO     | boxmot.utils.torch_utils:select_device:78 - Yolo Tracking v15.0.1 🚀 Python-3.10.12 torch-2.7.1+cu126
CUDA:0 (NVIDIA GeForce RTX 3070 Laptop GPU, 8192MiB)
2025-08-26 11:51:19.923 | ERROR    | boxmot.appearance.backends.base_backend:download_model:152 - Found existing ReID weights at osnet_x1_0_msmt17.pt; skipping download.
2025-08-26 11:51:20.352 | SUCCESS  | boxmot.appearance.reid.registry:load_pretrained_weights:64 - Loaded pretrained weights from osnet_x1_0_msmt17.pt


## Running Tracking on the Video (Simple Version, No Speed Control)

Here we run the full tracking-by-detection pipeline on a sample video for the first time:

```text
Frame → Detector (SSD) → Detections → tracker.update() → Tracks + IDs → Display
```

Key implementation notes:

* **Tracker input format**: BoxMOT expects detections as an array with columns `[x1, y1, x2, y2, confidence, class]`. Since our SSD model's output is already in this exact (`xyxy`) format, we just need to assemble these values; if a different detector produced a different format (e.g. `center_x, center_y, width, height`), we'd need to convert it first.
* **Combining CPU and GPU**: the image and the detector/Re-ID model run on the GPU (since neural-network computation is heavy), but the tracking logic itself (distance calculation, matching, track management) inside `tracker.update()` can run efficiently on the CPU. This combination is very reasonable and efficient for a practical pipeline.
* By simply calling `tracker.update(detections, frame)`, BoxMOT handles the cross-frame association internally, and `tracker.plot_results(...)` draws the bounding boxes and IDs on the frame.

⚠️ Note: in this version, the video is displayed without any timing management, so it may play faster or slower than real-time (due to the difference between the video's own FPS and the system's processing speed). We address this in the next section by computing an appropriate delay.


In [11]:
VIDEO_PATH = "football_players_detection/video/video2.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # (H, W, Ch) >>> (batch,Ch, H, W)
    img_tensor = torch.from_numpy(frame).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

    with torch.no_grad():
        preds = model(img_tensor)[0]

    detections = []
    if "boxes" in preds:
        boxes  = preds["boxes"].cpu().numpy()
        scores = preds["scores"].cpu().numpy()
        labels = preds["labels"].cpu().numpy()
        for (x1, y1, x2, y2), score, label in zip(boxes, scores, labels):
            detections.append([x1, y1, x2, y2, float(score), int(label)])
    detections = np.array(detections)

    tracks = tracker.update(detections, frame)

    if tracks.size > 0:
        tracker.plot_results(frame, show_trajectories=False)

    cv2.imshow("tracking", frame)
    if cv2.waitKey(1) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Managing FPS and Displaying the Video at Its Real Speed

We import `time` so we can measure how long processing each frame takes.


In [12]:
import time

### Improved Version: Displaying the Video at Its Actual FPS

The problem with the previous version was that the system's processing speed (which can be much higher than the video's original FPS) made the video play faster or slower than real-time. To fix this:

1. We read the video's actual frame rate with `cap.get(cv2.CAP_PROP_FPS)` (`target_fps`, 25 here) and use it to compute the required time between frames (`target_duration_ms`).
2. For each frame, we record the start and end time of processing (detection + tracking), and use it to compute and display the instantaneous processing FPS on the frame.
3. We compute the required delay as `target_duration_ms − processing_time_ms`, and if it comes out negative (i.e. processing was slower than the video's FPS), we clamp it to a minimum value (1 ms) so the delay is never negative:

```python
wait_time = target_duration_ms - processing_time_ms
if wait_time < 1:
    wait_time = 1
```

This way, `cv2.waitKey(int(wait_time))` makes the video play at approximately its real speed (25 FPS), even when the system processes it faster.


In [15]:
VIDEO_PATH = "football_players_detection/video/video2.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

target_fps = cap.get(cv2.CAP_PROP_FPS)
target_duration_ms = int((1 / target_fps) * 1000)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Record the start time for frame processing
    start_time = time.time()
    
    # (H, W, Ch) >>> (batch,Ch, H, W)
    img_tensor = torch.from_numpy(frame).permute(2,0,1).unsqueeze(0).float().to(device) / 255.0

    with torch.no_grad():
        preds = model(img_tensor)[0]

    detections = []
    if "boxes" in preds:
        boxes  = preds["boxes"].cpu().numpy()
        scores = preds["scores"].cpu().numpy()
        labels = preds["labels"].cpu().numpy()
        for (x1, y1, x2, y2), score, label in zip(boxes, scores, labels):
            detections.append([x1, y1, x2, y2, float(score), int(label)])
    detections = np.array(detections)

    tracks = tracker.update(detections, frame)

    if tracks.size > 0:
        tracker.plot_results(frame, show_trajectories=False)

    # Record the end time of processing
    end_time = time.time()
    
    # Calculate and display the FPS
    duration = end_time - start_time
    fps = 1 / duration
    cv2.putText(frame, f"FPS: {int(fps)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    print(fps)

    
    # Calculate the required delay
    processing_time_ms = (end_time - start_time) * 1000
    wait_time = target_duration_ms - processing_time_ms
    if wait_time < 1:
        wait_time = 1
        
    cv2.imshow("tracking", frame)
    if cv2.waitKey(int(wait_time)) == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

17.110319703997423
20.333456146134306
23.07935774263484
25.379724316539797
28.160275002685573
29.11619253890902
28.607995198242993
24.28088294035579
18.439824319987338
30.760854259563484
25.212820697780664
27.926281026952168
23.14456302214963
24.121831147918105
25.742665651928412
25.400627399682666
24.51533412434464
25.64241390483527
21.705378859230585
26.688623478432394
24.485708948253315
20.01070595363616
24.213320402025136
27.35956895820695
23.66521285299179
22.561423515109787
24.75830234342719
20.586955668119526
28.03510483994947
25.99506662534862
26.045915484211505
21.2287057704085
27.23574025974026
30.27460264757258
24.41004033126342
21.19887796618736
22.95104787961696
22.0250901892004
28.693912734138767
31.32625792622357
25.308207978084706
28.243520420187874
30.74259160173858
27.46599087152689
30.109431307518914
28.615802364691998
16.390786810163583
26.07052329954066
26.71412102645105
23.1291199550024
26.702386106088774
24.223248937349844
27.484348686495377
25.857087373852575
28

Checking the actual FPS of the input video (used in the delay calculation above).

---

In the next notebook (`04-ByteTrack.ipynb`) we run the same pipeline with the **ByteTrack** algorithm (no Re-ID, motion-only) and compare the results against BoT-SORT.


In [16]:
target_fps

25.0